In [ ]:
# ---------- CELL 1: INSTALL DEPENDENCIES ----------
import sys
import subprocess
import os

# Force install missing packages into the kernel's environment
packages = ["google-genai", "pydantic", "python-dotenv", "google-api-python-client", "google-auth-oauthlib", "requests"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"✅ {pkg} installed")

# ---------- SETUP PATH AND ENV ----------
from pathlib import Path

# Determine project root
cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd

os.chdir(str(project_root))
sys.path.insert(0, str(project_root))

# Ensure agents/__init__.py exists
init_file = project_root / "agents" / "__init__.py"
init_file.parent.mkdir(parents=True, exist_ok=True)
if not init_file.exists():
    init_file.touch()

# Load .env
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# ---------- IMPORT YOUR AGENT ----------
# Your file is gen_agent.py (singular)
from agents.gen_agent import run_gen_ai_agent
from services.retriever import retrieve_context

print("\n✅ All imports successful!")

# ---------- TEST CASES ----------
test_cases = [
    ("What are your business hours?", "Direct"),
    ("Do you build mobile apps?", "Direct"),
    ("How long is free technical support?", "Direct"),
    ("What is the refund policy after 30 days?", "Indirect/Partial"),
    ("Do you offer discounts for startups?", "Not in brochure"),
]

print("\n" + "="*60)
print("BATCH EVALUATION")
print("="*60)

for query, category in test_cases:
    email = f"Subject: Test\n\n{query}"
    try:
        result = run_gen_ai_agent(email)
        print(f"\nQ: {query[:50]}...")
        print(f"  Category: {category}")
        print(f"  Confidence: {result['confidence_score']}/10 → {result['status']}")
        print(f"  Draft (first 100 chars): {result['draft_reply'][:100]}...")
    except Exception as e:
        print(f"\nQ: {query[:50]}...")
        print(f"  ❌ ERROR: {e}")
    print("-" * 50)